# Fine-tune Qwen2.5 for Insurance Claims Classification (Unsloth + QLoRA → Ollama)

This notebook turns your local **Qwen2.5** into an insurance **claims classification chatbot**.

**What happens, in order**

| # | Step | What it does |
|---|------|--------------|
| 1 | Setup | Install Unsloth, check the GPU |
| 2 | Data | Load `claims_train.jsonl` (440 rows) and `claims_test.jsonl` (110 rows) |
| 3 | Baseline | Test your **current** Ollama Qwen2.5 on the test set (score *before* fine-tuning) |
| 4 | Load model | Load Qwen2.5 in **4-bit** (QLoRA: frozen base model compressed to 4-bit) |
| 5 | LoRA | Add the trainable **A and B bridges** (rank `r`) — everything else stays frozen |
| 6 | Train | Epochs / steps / learning rate — only A and B are updated |
| 7 | Evaluate | Score the fine-tuned model on the **unseen** test set |
| 8 | Export | Merge the bridges into the weights → **GGUF** → `ollama create qwen2.5-claims` |
| 9 | Test in Ollama | Score the new Ollama model and chat with it |

**Output format the model learns** (one JSON object per customer message):
```json
{"claim_category": "Motor - Collision", "priority": "High",
 "reason": "Road accident involving an injured passenger.",
 "next_step": "Share photos of the damage, ..."}
```

**11 categories:** Motor - Collision, Motor - Theft, Health - Hospitalization, Health - Outpatient, Property - Fire, Property - Water Damage, Property - Burglary, Life - Death Claim, Travel, Liability - Third Party, General Inquiry.

> **Why can't we fine-tune the Ollama model file directly?** Ollama stores models as **GGUF** (already quantized, inference-only). Training needs the original Hugging Face weights of the *same* model. So we train from the matching Hugging Face checkpoint, then convert the result back to GGUF and import it into Ollama as a **new** model. Your existing `qwen2.5` in Ollama is not changed.

### Requirements
- **NVIDIA GPU with CUDA** (Unsloth training needs it). 6 GB+ VRAM is enough for 0.5B–3B; ~16 GB for 7B.
  - No NVIDIA GPU (e.g. Mac or CPU-only)? Run sections 1–8 on **Google Colab** (free T4 GPU), download the `.gguf` file, then run section 9 on your machine.
- **Ollama** running locally (`ollama serve`) for the baseline (section 3) and the final test (section 9).
- Put `claims_train.jsonl` and `claims_test.jsonl` in the same folder as this notebook.

## 1. Setup
Run once. On Windows, use **WSL2**. If you already have a working Unsloth environment, skip the install cell.

In [ ]:
# Install Unsloth (pulls in torch, transformers, trl, peft, bitsandbytes, datasets)
%pip install -q unsloth
%pip install -q scikit-learn matplotlib requests

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No CUDA GPU found. Training (sections 4-8) needs an NVIDIA GPU - use Google Colab for those.")

## 2. Configuration
**Match `BASE_MODEL` to the Qwen2.5 size you have in Ollama** (check with `ollama list`):

| Your Ollama model | `OLLAMA_BASE_MODEL` | `BASE_MODEL` (Hugging Face, 4-bit) |
|---|---|---|
| `qwen2.5:0.5b` | `qwen2.5:0.5b` | `unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit` |
| `qwen2.5:1.5b` | `qwen2.5:1.5b` | `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit` |
| `qwen2.5:3b` | `qwen2.5:3b` | `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` |
| `qwen2.5` / `qwen2.5:7b` | `qwen2.5:7b` | `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` |

In [ ]:
# ---- Model ----
BASE_MODEL        = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"   # Hugging Face weights to train
OLLAMA_BASE_MODEL = "qwen2.5:1.5b"                              # your existing Ollama model (for the baseline)
NEW_OLLAMA_MODEL  = "qwen2.5-claims"                            # name of the fine-tuned model in Ollama
OLLAMA_URL        = "http://localhost:11434"

# ---- Data ----
TRAIN_FILE = "claims_train.jsonl"
TEST_FILE  = "claims_test.jsonl"

# ---- LoRA (the A and B bridges) ----
LORA_RANK  = 16      # r = number of tiny nodes in each bridge (width of the gate)
LORA_ALPHA = 16      # scaling of the bridge's correction (usually = r)

# ---- Training ----
MAX_SEQ_LENGTH = 1024
EPOCHS         = 3       # full passes over the 440 training examples
BATCH_SIZE     = 4       # examples per forward pass
GRAD_ACCUM     = 2       # batches combined before one update -> effective batch = 8
LEARNING_RATE  = 2e-4    # size of each correction (typical for LoRA/QLoRA)

# ---- Output ----
OUTPUT_DIR   = "qwen25_claims"
GGUF_QUANT   = "q4_k_m"  # 4-bit GGUF for Ollama (good size/quality balance)
EVAL_LIMIT   = None      # set e.g. 30 for a quick check; None = all 110 test rows

## 3. Load and inspect the dataset

In [ ]:
import json, os, collections

for f in (TRAIN_FILE, TEST_FILE):
    if not os.path.exists(f):
        raise FileNotFoundError(f"{f} not found. Put it next to this notebook (on Colab: upload via the Files panel).")

def load_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]

train_rows = load_jsonl(TRAIN_FILE)
test_rows  = load_jsonl(TEST_FILE)

SYSTEM_PROMPT = train_rows[0]["messages"][0]["content"]

def expected(row):
    return json.loads(row["messages"][-1]["content"])

def user_text(row):
    return row["messages"][1]["content"]

CATEGORIES = sorted({expected(r)["claim_category"] for r in train_rows})

print(f"Train rows: {len(train_rows)} | Test rows: {len(test_rows)}")
print("\nTrain label counts:")
for k, v in sorted(collections.Counter(expected(r)["claim_category"] for r in train_rows).items()):
    print(f"  {k:<26} {v}")

print("\nExample:")
print("USER     :", user_text(train_rows[0]))
print("ASSISTANT:", train_rows[0]["messages"][-1]["content"])

### Helpers: parse the model's answer and score it
The model should answer with JSON. If it adds extra text, we pull out the first `{...}` block; if that fails, we look for a category name in the text.

In [ ]:
import re

def parse_prediction(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            data = json.loads(match.group(0))
            return {
                "claim_category": str(data.get("claim_category", "")).strip(),
                "priority": str(data.get("priority", "")).strip(),
                "valid_json": True,
            }
        except json.JSONDecodeError:
            pass
    lowered = text.lower()
    category = next((c for c in CATEGORIES if c.lower() in lowered), "UNPARSED")
    priority = next((p for p in ("High", "Medium", "Low") if p.lower() in lowered), "UNPARSED")
    return {"claim_category": category, "priority": priority, "valid_json": False}


def evaluate(predict_fn, rows, limit=None, show_errors=5, title=""):
    rows = rows[:limit] if limit else rows
    y_true, y_pred, results = [], [], []
    cat_ok = pri_ok = json_ok = 0
    for i, row in enumerate(rows, 1):
        gold = expected(row)
        raw = predict_fn(user_text(row))
        pred = parse_prediction(raw)
        cat_ok += pred["claim_category"] == gold["claim_category"]
        pri_ok += pred["priority"] == gold["priority"]
        json_ok += pred["valid_json"]
        y_true.append(gold["claim_category"]); y_pred.append(pred["claim_category"])
        results.append({"message": user_text(row), "gold": gold, "pred": pred, "raw": raw})
        if i % 10 == 0 or i == len(rows):
            print(f"\r  {i}/{len(rows)} done", end="")
    n = len(rows)
    summary = {"category_acc": cat_ok / n, "priority_acc": pri_ok / n, "valid_json": json_ok / n}
    print(f"\n{title} Category accuracy: {summary['category_acc']:.1%} | "
          f"Priority accuracy: {summary['priority_acc']:.1%} | Valid JSON: {summary['valid_json']:.1%}")
    errors = [r for r in results if r["pred"]["claim_category"] != r["gold"]["claim_category"]]
    for r in errors[:show_errors]:
        print(f"\n  ✗ {r['message']}\n    expected: {r['gold']['claim_category']} | got: {r['pred']['claim_category']}")
    return summary, y_true, y_pred, results

## 4. Baseline: your current Ollama Qwen2.5 (before fine-tuning)
This tells us how good the model is **before** training, with the exact same instructions. Skipped automatically if Ollama isn't reachable (e.g. on Colab).

In [ ]:
import requests

def ollama_available():
    try:
        return requests.get(f"{OLLAMA_URL}/api/tags", timeout=3).ok
    except requests.RequestException:
        return False

def ollama_models():
    return [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json().get("models", [])]

def ask_ollama(model, message, system=SYSTEM_PROMPT):
    resp = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": model,
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": message}],
            "stream": False,
            "options": {"temperature": 0},
        },
        timeout=300,
    )
    resp.raise_for_status()
    return resp.json()["message"]["content"]

baseline = None
if ollama_available():
    print("Ollama models:", ollama_models())
    baseline, *_ = evaluate(lambda m: ask_ollama(OLLAMA_BASE_MODEL, m), test_rows,
                            limit=EVAL_LIMIT, title="[BASELINE Ollama]")
else:
    print("Ollama not reachable - skipping baseline. Start it with `ollama serve` to compare before vs after.")

## 5. Load Qwen2.5 in 4-bit (QLoRA)
`load_in_4bit=True` compresses the **frozen** base weights to 4-bit to save GPU memory. These weights will **not** be trained.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,            # auto: bfloat16 on newer GPUs, float16 on older (e.g. T4)
    load_in_4bit=True,     # QLoRA: frozen base model stored in 4-bit
)

## 6. Add LoRA bridges (A and B)
- `r` = **rank** = number of tiny nodes in each bridge.
- `target_modules` = **which connections get a bridge**: here the attention projections (`q/k/v/o_proj`) and the feed-forward projections (`gate/up/down_proj`) in every layer.
- Everything else is frozen. Watch the printed **trainable %**.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",                          # biases stay frozen too
    use_gradient_checkpointing="unsloth", # saves memory
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable (A and B only): {trainable:,} / {total:,} = {100 * trainable / total:.2f}%")

## 7. Format the data with Qwen's chat template
Each row becomes one text string in Qwen's format (`<|im_start|>system ... <|im_start|>user ... <|im_start|>assistant ...`).

In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def to_text(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)}

train_ds = Dataset.from_list(train_rows).map(to_text, remove_columns=["messages"])
print(train_ds[0]["text"])

## 8. Train
- **Step** = one batch → forward pass → loss → backward pass → update A and B.
- **Epoch** = all batches done once.
- **Learning rate** = how big each update is.
- `train_on_responses_only` makes the loss count **only the assistant's JSON answer**, not the system prompt or the customer's message, so the model learns *to answer*, not to copy questions.

In [ ]:
import inspect, math
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

config_kwargs = dict(
    dataset_text_field="text",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=10,
    lr_scheduler_type="linear",   # learning rate decreases slowly towards the end
    optim="adamw_8bit",
    weight_decay=0.01,            # L2-style regularisation on the trainable weights
    logging_steps=5,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    seed=42,
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    report_to="none",
)
# TRL renamed max_seq_length -> max_length in newer versions; support both.
cfg_params = inspect.signature(SFTConfig.__init__).parameters
config_kwargs["max_length" if "max_length" in cfg_params else "max_seq_length"] = MAX_SEQ_LENGTH
args = SFTConfig(**config_kwargs)

try:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, args=args)
except TypeError:  # newer TRL uses processing_class instead of tokenizer
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=train_ds, args=args)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

steps_per_epoch = math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM))
print(f"Examples: {len(train_ds)} | Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Steps per epoch: {steps_per_epoch} | Epochs: {EPOCHS} | Total update steps: ~{steps_per_epoch * EPOCHS}")

In [ ]:
stats = trainer.train()
print(f"\nTraining time: {stats.metrics['train_runtime'] / 60:.1f} min | Final loss: {stats.metrics['train_loss']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

log = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
if log:
    steps, losses = zip(*log)
    plt.figure(figsize=(7, 3.5))
    plt.plot(steps, losses, marker="o", markersize=3)
    plt.xlabel("Step"); plt.ylabel("Training loss"); plt.title("Loss goes down as A and B learn")
    plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 9. Evaluate the fine-tuned model on the unseen test set
The test messages come from **sentence templates the model never saw** during training, so this measures real generalisation, not memorisation.

In [ ]:
FastLanguageModel.for_inference(model)  # faster generation mode

def ask_finetuned(message, max_new_tokens=256):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": message}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(ask_finetuned("My scooter was hit by a car at a signal this morning and my leg is fractured."))

In [ ]:
finetuned, y_true, y_pred, ft_results = evaluate(ask_finetuned, test_rows, limit=EVAL_LIMIT, title="[FINE-TUNED]")

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

print(classification_report(y_true, y_pred, zero_division=0))

labels = sorted(set(y_true) | set(y_pred))
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=labels, xticks_rotation=60,
                                        ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Fine-tuned model: confusion matrix (test set)")
plt.tight_layout(); plt.show()

### Try your own messages (before exporting)

In [ ]:
my_questions = [
    "Hi, my bike got stolen from outside the gym last night. What do I do?",
    "My mother is admitted in hospital since yesterday with high fever and low platelets.",
    "A customer slipped on the wet floor of my bakery and broke her ankle.",
    "How do I download my policy document?",
    "Heavy rain flooded my basement and the washing machine and fridge are ruined.",
]
for q in my_questions:
    print("Q:", q)
    print("A:", ask_finetuned(q), "\n")

## 10. Save the adapter and export to GGUF
1. Save the **LoRA adapter** (just A and B — a few MB to tens of MB).
2. **Merge** A and B into the base weights (`W_new = W + B×A`) and convert to **GGUF** (`q4_k_m`) for Ollama.

The GGUF step downloads and builds `llama.cpp` the first time, so it can take several minutes.

In [ ]:
adapter_dir = f"{OUTPUT_DIR}/lora_adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("LoRA adapter saved to", adapter_dir)

gguf_dir = f"{OUTPUT_DIR}/gguf"
model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method=GGUF_QUANT)

In [ ]:
import glob

# Unsloth's GGUF file name/location differs between versions, so search for the newest .gguf
gguf_files = sorted(glob.glob(f"{OUTPUT_DIR}*/**/*.gguf", recursive=True) + glob.glob(f"{OUTPUT_DIR}*/*.gguf"),
                    key=os.path.getmtime)
gguf_files = [f for f in gguf_files if GGUF_QUANT.lower() in os.path.basename(f).lower()] or gguf_files
assert gguf_files, "No .gguf file found - check the output of the previous cell."
GGUF_PATH = os.path.abspath(gguf_files[-1])
print("GGUF file:", GGUF_PATH, f"({os.path.getsize(GGUF_PATH) / 1e9:.2f} GB)")

## 11. Create the Ollama model
We write a **Modelfile** that points to the GGUF, uses Qwen's chat format, bakes in the system prompt and sets `temperature 0` (classification should be deterministic).

> **Trained on Colab?** Download the `.gguf` and `Modelfile` from `qwen25_claims/`, put them in one folder on your machine, change the `FROM` line to the local `.gguf` file name, and run:
> `ollama create qwen2.5-claims -f Modelfile`

In [ ]:
import shutil, subprocess

Q3 = '"' * 3   # triple quote used by Modelfile multi-line values
chat_template = (
    "{{ if .System }}<|im_start|>system\n{{ .System }}<|im_end|>\n{{ end }}"
    "{{ if .Prompt }}<|im_start|>user\n{{ .Prompt }}<|im_end|>\n{{ end }}"
    "<|im_start|>assistant\n{{ .Response }}<|im_end|>\n"
)
modelfile = "\n".join([
    f"FROM {GGUF_PATH}",
    "",
    f"TEMPLATE {Q3}{chat_template}{Q3}",
    "",
    f"SYSTEM {Q3}{SYSTEM_PROMPT}{Q3}",
    "",
    "PARAMETER temperature 0",
    "PARAMETER num_ctx 2048",
    'PARAMETER stop "<|im_end|>"',
    'PARAMETER stop "<|im_start|>"',
]) + "\n"

MODELFILE_PATH = os.path.abspath(f"{OUTPUT_DIR}/Modelfile")
with open(MODELFILE_PATH, "w", encoding="utf-8") as fh:
    fh.write(modelfile)
print(modelfile)

if shutil.which("ollama"):
    subprocess.run(["ollama", "create", NEW_OLLAMA_MODEL, "-f", MODELFILE_PATH], check=True)
    print(f"\n✅ Created Ollama model '{NEW_OLLAMA_MODEL}'. Try:  ollama run {NEW_OLLAMA_MODEL}")
else:
    print("\n'ollama' command not found on this machine. Copy the .gguf and Modelfile to your computer and run:")
    print(f"  ollama create {NEW_OLLAMA_MODEL} -f Modelfile")

## 12. Test the new Ollama model
Scores the **Ollama** version of the fine-tuned model on the same test set and compares it with the baseline from section 4.

In [ ]:
if ollama_available() and any(m.startswith(NEW_OLLAMA_MODEL) for m in ollama_models()):
    ollama_ft, *_ = evaluate(lambda m: ask_ollama(NEW_OLLAMA_MODEL, m), test_rows,
                             limit=EVAL_LIMIT, title="[FINE-TUNED in Ollama]")
    print("\n               Category acc   Priority acc   Valid JSON")
    if baseline:
        print(f"Before (base)  {baseline['category_acc']:>11.1%}   {baseline['priority_acc']:>11.1%}   {baseline['valid_json']:>9.1%}")
    print(f"After (LoRA)   {ollama_ft['category_acc']:>11.1%}   {ollama_ft['priority_acc']:>11.1%}   {ollama_ft['valid_json']:>9.1%}")
else:
    print(f"Ollama model '{NEW_OLLAMA_MODEL}' not found yet - create it first (section 11).")

### Chat with the fine-tuned model through Ollama

In [ ]:
def classify(message, model_name=NEW_OLLAMA_MODEL):
    raw = ask_ollama(model_name, message)
    try:
        return json.loads(re.search(r"\{.*\}", raw, re.DOTALL).group(0))
    except Exception:
        return {"raw_output": raw}

print(json.dumps(classify("Someone broke into my flat while I was at work and took my laptop and gold chain."), indent=2))

In [ ]:
# Interactive loop - type a customer message, press Enter. Type 'quit' to stop.
while True:
    msg = input("\nCustomer message (or 'quit'): ").strip()
    if msg.lower() in {"quit", "exit", "q", ""}:
        break
    print(json.dumps(classify(msg), indent=2))

## Using the model outside this notebook
```bash
ollama run qwen2.5-claims "My car was stolen from the mall parking last night"
```
```python
import requests
r = requests.post("http://localhost:11434/api/chat", json={
    "model": "qwen2.5-claims", "stream": False,
    "messages": [{"role": "user", "content": "My father passed away last week, he had a policy with you."}]})
print(r.json()["message"]["content"])
```
The system prompt is baked into the Modelfile, so you only need to send the customer's message.

### Tuning tips
- **Accuracy low?** Increase `EPOCHS` to 4–5 or `LORA_RANK` to 32, or add more varied training examples.
- **Training loss near 0 but test accuracy drops?** Overfitting: reduce `EPOCHS` or `LEARNING_RATE`.
- **Out of memory?** Lower `BATCH_SIZE` to 2 and raise `GRAD_ACCUM` to 4 (same effective batch).
- **Try DoRA:** add `use_dora=True` to `get_peft_model` (slower training, often a bit more accurate).